# Лабораторна робота №2. Частина 1
**Тема:** Наука про дані: підготовчий етап. Аналіз VHI-індексу.

In [1]:
import os
import urllib.request
from datetime import datetime
import pandas as pd

## Завдання 2: Завантаження даних
Для кожної з адміністративних одиниць України завантажити структуровані файли VHI-індексу. 
Вимоги: додати дату/час у назву, реалізувати механізм запобігання повторного завантаження, `provinceID=0` ігнорується.

In [2]:
def download_vhi_data(folder="data"):
    if not os.path.exists(folder):
        os.makedirs(folder)

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for province_id in range(1, 28):
        existing_files = [f for f in os.listdir(folder) if f.startswith(f"vhi_id_{province_id}_")]
        if existing_files:
            print(f"Дані для NOAA ID {province_id} вже завантажені.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        filename = os.path.join(folder, f"vhi_id_{province_id}_{now}.csv")
        
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"Завантажено: NOAA ID {province_id} -> {filename}")
        except Exception as e:
            print(f"Помилка завантаження ID {province_id}: {e}")

download_vhi_data()

Дані для NOAA ID 1 вже завантажені.
Дані для NOAA ID 2 вже завантажені.
Дані для NOAA ID 3 вже завантажені.
Дані для NOAA ID 4 вже завантажені.
Дані для NOAA ID 5 вже завантажені.
Дані для NOAA ID 6 вже завантажені.
Дані для NOAA ID 7 вже завантажені.
Дані для NOAA ID 8 вже завантажені.
Дані для NOAA ID 9 вже завантажені.
Дані для NOAA ID 10 вже завантажені.
Дані для NOAA ID 11 вже завантажені.
Дані для NOAA ID 12 вже завантажені.
Дані для NOAA ID 13 вже завантажені.
Дані для NOAA ID 14 вже завантажені.
Дані для NOAA ID 15 вже завантажені.
Дані для NOAA ID 16 вже завантажені.
Дані для NOAA ID 17 вже завантажені.
Дані для NOAA ID 18 вже завантажені.
Дані для NOAA ID 19 вже завантажені.
Дані для NOAA ID 20 вже завантажені.
Дані для NOAA ID 21 вже завантажені.
Дані для NOAA ID 22 вже завантажені.
Дані для NOAA ID 23 вже завантажені.
Дані для NOAA ID 24 вже завантажені.
Дані для NOAA ID 25 вже завантажені.
Дані для NOAA ID 26 вже завантажені.
Дані для NOAA ID 27 вже завантажені.


## Завдання 3: Зчитування, очищення та зміна індексів
Зчитати файли у pandas dataframe. Здійснити data cleaning. 
Додати стовпчики з назвою та новим індексом області (заміна індексів NOAA на українську абетку, де 1 - Вінницька).

In [ ]:
def load_and_clean_data(folder="data"):
    noaa_to_ua = {
        24: 1, 25: 2, 5: 3, 6: 4, 27: 5, 23: 6, 26: 7, 7: 8, 
        11: 9, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 
        19: 16, 21: 17, 22: 18, 8: 19, 9: 20, 10: 21, 1: 22, 
        3: 23, 2: 24, 4: 25, 12: 26, 20: 27
    }
    
    area_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }

    df_list = []
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    
    for filename in os.listdir(folder):
        if filename.endswith(".csv"):
            filepath = os.path.join(folder, filename)
            old_id = int(filename.split('_')[2])
            new_id = noaa_to_ua.get(old_id, old_id)
            
            df_temp = pd.read_csv(filepath, header=1, names=headers, index_col=False)
            
            if 'empty' in df_temp.columns:
                df_temp = df_temp.drop(columns=['empty'])
            df_temp = df_temp.dropna()

            df_temp['Year'] = df_temp['Year'].astype(str).str.extract(r'(\d+)').astype(float)
            df_temp = df_temp.dropna(subset=['Year']) 
            df_temp['Year'] = df_temp['Year'].astype(int)
            
            df_temp['Area'] = new_id
            df_temp['Area_Name'] = area_names.get(new_id, 'Невідомо')
            
            df_temp = df_temp[df_temp['VHI'] >= 0]
            
            df_list.append(df_temp)
            
    return pd.concat(df_list, ignore_index=True)

df = load_and_clean_data()
print("Дані очищено та підготовлено. Вивід перших 5 рядків:")
display(df.head())

Дані очищено та підготовлено. Перші 5 рядків:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area,Area_Name
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


## Завдання 4.1: Ряд VHI для області за вказаний рік
Процедура формує вибірку значень VHI по тижнях для заданої області та року.

In [4]:
def get_vhi_series_by_year(dataframe, area_index, year):
    result = dataframe[(dataframe['Area'] == area_index) & (dataframe['Year'] == year)][['Week', 'VHI']]
    
    area_name = dataframe[dataframe['Area'] == area_index]['Area_Name'].iloc[0] if not result.empty else "Невідома область"
    print(f"Ряд VHI для області: {area_name} за {year} рік")
    
    return result

display(get_vhi_series_by_year(df, 15, 2020))

Ряд VHI для області: Полтавська за 2020 рік


,Week,VHI
19414,1.0,43.53
19415,2.0,44.00
19416,3.0,44.41
19417,4.0,44.99
19418,5.0,45.05
19419,6.0,43.83
19420,7.0,40.25
19421,8.0,37.56
19422,9.0,36.75
19423,10.0,36.15


## Завдання 4.2: Ряд VHI за вказаний діапазон років для вказаних областей

In [5]:
def get_vhi_for_areas_and_years(dataframe, area_indices, start_year, end_year):
    result = dataframe[
        (dataframe['Area'].isin(area_indices)) & 
        (dataframe['Year'] >= start_year) & 
        (dataframe['Year'] <= end_year)
    ][['Year', 'Week', 'Area_Name', 'VHI']]
    
    print(f"Дані VHI для областей з індексами {area_indices} за {start_year}-{end_year} роки")
    return result

display(get_vhi_for_areas_and_years(df, [1, 9], 2015, 2016)) 

Дані VHI для областей з індексами [1, 9] за 2015-2016 роки


,Year,Week,Area_Name,VHI
3852,2015,1.0,Київська,46.27
3853,2015,2.0,Київська,47.79
3854,2015,3.0,Київська,48.26
3855,2015,4.0,Київська,47.77
3856,2015,5.0,Київська,46.46
...,...,...,...,...
34555,2016,48.0,Вінницька,47.38
34556,2016,49.0,Вінницька,49.51
34557,2016,50.0,Вінницька,49.57
34558,2016,51.0,Вінницька,48.30


## Завдання 4.3: Пошук екстремумів (min та max), середнього, медіани
Статистика для вказаної області та року.

In [6]:
def get_vhi_statistics(dataframe, area_index, year):
    subset = dataframe[(dataframe['Area'] == area_index) & (dataframe['Year'] == year)]['VHI']
    
    if subset.empty:
        print("Дані за вказаними параметрами відсутні.")
        return
        
    area_name = dataframe[dataframe['Area'] == area_index]['Area_Name'].iloc[0]
    
    print(f"Статистика VHI для області: {area_name} ({year} рік)")
    print(f"Мінімум (min): {subset.min():.2f}")
    print(f"Максимум (max): {subset.max():.2f}")
    print(f"Середнє (mean): {subset.mean():.2f}")
    print(f"Медіана (median): {subset.median():.2f}")

get_vhi_statistics(df, 12, 2005)

Статистика VHI для області: Львівська (2005 рік)
Мінімум (min): 43.52
Максимум (max): 61.93
Середнє (mean): 53.45
Медіана (median): 54.56
